# Debug ChatGroq Initialization Error

Este notebook diagnostica a falha de inicialização do `ChatGroq` no `blog-agent/agent.py`, valida variáveis de ambiente e propõe uma forma mais robusta de inicializar o modelo Groq.

## 1. Import Required Libraries and Setup

Importando bibliotecas essenciais para depuração, incluindo `os`, `dotenv`, `langchain_groq` e `logging`. Carrega variáveis de ambiente e configura logs para capturar erros detalhados.

In [ ]:
import os
import logging
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

print('Python executable:', os.sys.executable)
print('Working directory:', os.getcwd())
print('GROQ_API_KEY loaded:', bool(os.getenv('GROQ_API_KEY')))
print('TOKEN loaded:', bool(os.getenv('TOKEN')))

## 2. Understanding the Error Stack Trace

A falha acontece durante a validação de ambiente do `ChatGroq`, possivelmente ao criar o cliente `groq.Groq(...)`.
Isso pode ser causado por:

- `GROQ_API_KEY` inválida ou ausente
- nome de modelo não suportado
- versão da biblioteca incompatível
- parâmetros ausentes ou incorretos

In [ ]:
import inspect
import arxiv
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage
from git import Repo

print('Imports successful')
print('ChatGroq location:', inspect.getsourcefile(ChatGroq))
print('ChatGroq init signature:', inspect.signature(ChatGroq.__init__))

## 3. Validate Environment Variables

Verificar se `GROQ_API_KEY` está corretamente carregada sem espaços extras ou quebras de linha.

In [ ]:
groq_key = os.getenv('GROQ_API_KEY')
print('Raw GROQ_API_KEY:', repr(groq_key))
if groq_key:
    print('Trimmed key:', repr(groq_key.strip()))
    if groq_key != groq_key.strip():
        logger.warning('GROQ_API_KEY contains whitespace or hidden characters')
else:
    logger.error('GROQ_API_KEY is not set')

print('TOKEN env:', repr(os.getenv('TOKEN')))

## 4. Initialize ChatGroq with Error Handling

Tenta criar o `ChatGroq` com tratamento de exceção para capturar a causa exata do erro. Também validamos o nome do modelo `llama-3.1-70b-versatile`.

In [ ]:
def create_chatgroq(api_key: str, model: str = 'llama-3.1-70b-versatile'):
    try:
        llm = ChatGroq(
            api_key=api_key,
            model=model,
            temperature=0.5,
        )
        logger.info('ChatGroq created successfully')
        return llm
    except Exception as e:
        logger.error('Failed to create ChatGroq: %s', e)
        raise

api_key = os.getenv('GROQ_API_KEY')
llm = create_chatgroq(api_key) if api_key else None

## 5. Test LLM Connection and Functionality

Executar uma chamada simples para verificar se o modelo responde corretamente.

In [ ]:
if llm is not None:
    try:
        response = llm.invoke([HumanMessage(content='Responda com uma palavra: teste')])
        print('Response:', response.content)
    except Exception as e:
        logger.error('LLM invocation failed: %s', e)
        raise
else:
    logger.error('LLM not initialized; skipping invocation')

## 6. Implement Retry Logic and Fallbacks

Adiciona lógica de retry com backoff para falhas transitórias e um fallback mais robusto para inicialização do agente.

In [ ]:
import time

def retry_create_chatgroq(api_key: str, model: str = 'llama-3.1-70b-versatile', retries: int = 3, backoff: float = 2.0):
    for attempt in range(1, retries + 1):
        try:
            return create_chatgroq(api_key=api_key, model=model)
        except Exception as exc:
            if attempt == retries:
                logger.error('Final attempt failed after %s retries', retries)
                raise
            logger.warning('Attempt %s failed: %s. Retrying in %s seconds...', attempt, exc, backoff)
            time.sleep(backoff)
            backoff *= 2

if groq_key:
    llm = retry_create_chatgroq(groq_key)
    logger.info('Retry mechanism completed')

## Conclusão

Se a inicialização do `ChatGroq` ainda falhar, provavelmente o problema é:

- chave `GROQ_API_KEY` inválida
- modelo não disponível na conta
- limite da API
- incompatibilidade entre `langchain-groq` e `groq`

Use as mensagens de erro capturadas acima para ajustar a configuração ou atualizar a inicialização no `agent.py`.